# Juggle counter with YOLO

Here we count how many times a ball is juggled in a video.
We do not train anything. We reuse a pretrained YOLO model to find the ball, then we count the juggles with simple math.
Run each cell with Shift+Enter.

## 1. Imports

Here we import the libraries we need.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO
from scipy.signal import find_peaks

print('ok')

## 2. Load the model

Here we load the small YOLO model (yolov8s). The nano model was faster but it confused the ball with a teddy bear, so we do not use it.
In COCO the ball is class number 32 (name: sports ball).

In [ ]:
model = YOLO('yolov8s.pt')
SPORTS_BALL = 32
print(model.names[SPORTS_BALL])

## 3. Check one frame

Here we test the detection on a single frame before processing the whole video.

In [ ]:
VIDEO = 'jungle_video.mp4'   # use a sharp video, see the note in section 4

cap = cv2.VideoCapture(VIDEO)
fps = cap.get(cv2.CAP_PROP_FPS)
n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(n_frames, 'frames', round(fps, 1), 'fps', round(n_frames / fps, 1), 'seconds')

cap.set(cv2.CAP_PROP_POS_FRAMES, n_frames // 2)
ok, frame = cap.read()
cap.release()

res = model(frame, classes=[SPORTS_BALL], conf=0.25, verbose=False)[0]
plt.figure(figsize=(9, 6))
plt.imshow(cv2.cvtColor(res.plot(), cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.show()

## 4. Track the ball across the video

Here we run YOLO on every frame and store the ball height. The image y axis points down, so a high ball has a small y.
We store height as H minus y, so that a high ball gives a big number.
We also store the box, because we reuse it later to draw the annotated video.

This loop is the slow part. YOLO runs one forward pass per frame on the CPU, so it takes a few minutes.

Note: this works because the video is sharp. On a blurry video the ball becomes a white blob in the air and YOLO loses it.

In [ ]:
cap = cv2.VideoCapture(VIDEO)
H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

heights, boxes, times = [], [], []
i = 0
while True:
    ok, frame = cap.read()
    if not ok:
        break
    res = model(frame, classes=[SPORTS_BALL], conf=0.25, verbose=False)[0]
    b = res.boxes
    if len(b) > 0:
        j = int(b.conf.argmax())
        x1, y1, x2, y2 = b.xyxy[j].tolist()
        heights.append(H - (y1 + y2) / 2)
        boxes.append((x1, y1, x2, y2))
    else:
        heights.append(np.nan)
        boxes.append(None)
    times.append(i / fps)
    i += 1

cap.release()
heights = np.array(heights)
times = np.array(times)
found = int(np.sum(~np.isnan(heights)))
print('ball found in', found, 'of', len(heights), 'frames')

## 5. Plot the raw height

Here we plot the ball height over time. Each bump is the ball going up and coming back down, which is one juggle.

In [ ]:
plt.figure(figsize=(11, 4))
plt.plot(times, heights, '.-')
plt.xlabel('time (s)')
plt.ylabel('ball height (pixels)')
plt.title('raw ball height')
plt.grid(True, alpha=0.3)
plt.show()

## 6. Fill gaps and smooth

Here we clean the signal. First we fill the frames where the ball was not found (NaN) by linear interpolation.
Then we smooth with a small moving average, so that tiny detection jitter does not create fake bumps.
We keep the window small (3 frames). A larger window flattens real juggles and breaks the count.

In [ ]:
s = heights.copy()
nan = np.isnan(s)
if nan.any():
    s[nan] = np.interp(np.flatnonzero(nan), np.flatnonzero(~nan), s[~nan])

window = 3
s = np.convolve(s, np.ones(window) / window, mode='same')

plt.figure(figsize=(11, 4))
plt.plot(times, heights, '.', alpha=0.4, label='raw')
plt.plot(times, s, '-', label='smoothed')
plt.xlabel('time (s)')
plt.ylabel('ball height (pixels)')
plt.title('raw vs smoothed')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 7. The derivative (velocity)

Here is the core idea. The derivative of the height is the vertical velocity of the ball.

When the ball goes up, the height increases, so the velocity is positive.
At the top of a juggle, the ball stops for an instant, so the velocity is zero.
When the ball falls, the height decreases, so the velocity is negative.

So the top of a juggle is the moment where the velocity goes from positive to negative. That is a downward zero crossing.

Here we compute the derivative with np.gradient and plot it under the height so we can compare them.

In [ ]:
velocity = np.gradient(s)

fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
ax[0].plot(times, s)
ax[0].set_ylabel('height')
ax[0].set_title('height')
ax[0].grid(True, alpha=0.3)

ax[1].plot(times, velocity)
ax[1].axhline(0, color='red', linewidth=1)
ax[1].set_ylabel('velocity')
ax[1].set_xlabel('time (s)')
ax[1].set_title('velocity. it crosses zero downward at the top of each juggle')
ax[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Candidates from the derivative

Here we find every frame where the velocity goes from positive to negative. Those are all the tops in the signal.

In [ ]:
candidates = np.where((velocity[:-1] > 0) & (velocity[1:] <= 0))[0] + 1
print('candidate tops from the derivative:', len(candidates))

## 9. Keep only the real juggles (prominence)

The derivative finds too many tops. Some are not juggles: the ball being picked up at the start, small shoulders,
and the ball settling on the floor at the end. They are small bumps that barely rise above the dips next to them.

A real juggle is a bump that clearly stands out. The prominence of a peak is how far the signal drops on both sides
before it climbs higher again. Here we keep only peaks whose prominence is at least 15 percent of the total height range.
We also ask for a minimum time gap so we never count the same juggle twice.

find_peaks does exactly this: it looks for the same downward tops, then filters them by prominence and by distance.

In [ ]:
amplitude = s.max() - s.min()
min_gap = int(fps * 0.20)                  # at least 0.2 s between two juggles
prominence = 0.15 * amplitude              # a real bump stands out by at least 15 percent

peaks, _ = find_peaks(s, distance=min_gap, prominence=prominence)
print('real juggles:', len(peaks))

plt.figure(figsize=(12, 4))
plt.plot(times, s, color='steelblue')
plt.plot(times[candidates], s[candidates], 'rx', ms=9, label='all candidates')
plt.plot(times[peaks], s[peaks], 'go', ms=11, mfc='none', mew=2, label='real juggles')
plt.legend()
plt.title('candidates vs real juggles')
plt.grid(True, alpha=0.3)
plt.show()

## 10. Show the count

Here we plot the height with a red dot and a number on each juggle.
Check this count against the video. If it is wrong, change prominence or min_gap in section 9 and run it again.

In [ ]:
plt.figure(figsize=(11, 4))
plt.plot(times, s, '-')
plt.plot(times[peaks], s[peaks], 'ro', markersize=10)
for k, p in enumerate(peaks, 1):
    plt.annotate(str(k), (times[p], s[p]), textcoords='offset points', xytext=(0, 8), ha='center')
plt.xlabel('time (s)')
plt.ylabel('height')
plt.title(str(len(peaks)) + ' juggles')
plt.grid(True, alpha=0.3)
plt.show()

## 11. Detect breaks and count the streak

The count so far is every juggle in the whole video. But successive juggles means juggles in a row without dropping.
Here we split the juggles into streaks. A streak breaks in two cases.
First, the ball touches the ground: the lowest point between two juggles reaches the floor.
Second, there is a long pause: the gap between two juggles is much longer than the normal rhythm, for example the ball is held on the foot.

Note: in this sample video the ball never drops in the middle, so there is a single streak of 19.
On a video where the player drops the ball and starts again, this splits into several streaks.

In [ ]:
valleys, _ = find_peaks(-s, distance=min_gap, prominence=0.15 * amplitude)
floor = s.min()
typical_valley = np.median(s[valleys])                 # normal low point during juggling
drop_line = floor + 0.4 * (typical_valley - floor)     # below this, the ball is on the ground

intervals = np.diff(peaks) / fps
max_gap = 2.5 * np.median(intervals)                   # a pause longer than this breaks the streak

streaks = []
current = [int(peaks[0])]
for prev, cur in zip(peaks[:-1], peaks[1:]):
    ball_hit_floor = s[prev:cur].min() < drop_line
    long_pause = (cur - prev) / fps > max_gap
    if ball_hit_floor or long_pause:
        streaks.append(current)
        current = [int(cur)]
    else:
        current.append(int(cur))
streaks.append(current)

for i, st in enumerate(streaks, 1):
    print('streak', i, ':', len(st), 'juggles')
print('longest streak:', max(len(st) for st in streaks))

## 12. Annotated video

Here we write a new video with the ball box and a counter that goes up on each juggle.
We reuse the boxes stored in section 4, so YOLO does not run again. We shrink the output to keep the file small.

In [ ]:
peak_set = set(int(p) for p in peaks)
out_w = 960

cap = cv2.VideoCapture(VIDEO)
src_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
scale = out_w / src_w
out_size = (out_w, int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) * scale))
# avc1 is H.264. Note: mp4v fails silently on this OpenCV build, so we use avc1.
writer = cv2.VideoWriter('juggle_annotated.mp4', cv2.VideoWriter_fourcc(*'avc1'), fps, out_size)

count = 0
i = 0
while True:
    ok, frame = cap.read()
    if not ok:
        break
    if i in peak_set:
        count += 1
    frame = cv2.resize(frame, out_size)
    if boxes[i] is not None:
        x1, y1, x2, y2 = [int(v * scale) for v in boxes[i]]
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 3)
    cv2.putText(frame, 'Juggles: ' + str(count), (20, 50),
                cv2.FONT_HERSHEY_SIMPLEX, 1.4, (0, 255, 0), 3, cv2.LINE_AA)
    writer.write(frame)
    i += 1

cap.release()
writer.release()
print('saved juggle_annotated.mp4, final count:', count)